# Autoencoder denso: ejercicio guiado

Analizando lo procesado anteriormente y prestando atención a la representación del `latent_space` de batch de TEST ploteado al momento de analizar el encoder, realice un paneo en X e Y del `latent_space` y grafique el resultado que genera el decoder para cada número generado.


Objetivo: comparar dos autoencoders de dimensión latente 2 que solo difieren en la activación final del decoder (`Tanh()` vs `ReLU()`), y analizar cómo esa decisión impacta en:

- la reconstrucción de imágenes,
- la geometría del espacio latente,
- la generación de nuevas cifras a partir del decoder.



Para realizar esto, se deja armada una celda donde 
solamente debe fijar los siguientes hiper-parámetros:
- `n` número de imágenes (digits del NMIST) a representar (mosaico de nxn imágenes).
- `x_min` valor mínimo de la variable x del `latent_space`.
- `x_max` valor mínimo de la variable x del `latent_space`.
- `y_min` valor mínimo de la variable x del `latent_space`.
- `y_max` valor mínimo de la variable x del `latent_space`.


¿Cómo afecta los límites de x e y del `latent_space` con la 
última función de activación del encoder? 
Analice el comportamiento modificando dicha función y explique 
con sus palabras lo que sucede.

La convención de este notebook es la misma que en el notebook principal: cargar datos, definir el modelo, entrenar o cargar pesos, analizar el espacio latente y cerrar con una exploración generativa.

## 1. Utilidades comunes

Imports, carga del dataset y funciones auxiliares.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, datetime
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# librerías
import torch
from torch import nn
from torch.utils import data

import torchvision
from torchvision import datasets
import torchvision.transforms as transforms



import numpy as np
import matplotlib.pyplot as plt



# funciones

def load_mnist(BATCH_SIZE=32):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5), (0.5)) 
    ])

    trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    trainloader = data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)

    testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    testloader = data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)

    print(f'Train Data Shape: {trainset.train_data.numpy().shape}')
    print(f'Test Data Shape: {testset.test_data.numpy().shape}')

    return trainloader, testloader


def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

## 2. Ruta de trabajo

Definir `direccion` según el entorno de ejecución para guardar o cargar pesos entrenados.


In [ ]:
#from google.colab import drive
#drive.mount("/content/drive")


In [ ]:
# una ruta del drive...
direccion = '/content/drive/My Drive/CIA_marcos/deep_learning/clase_7/autoencoder'

In [ ]:
# esta ruta es el colab mismo.. al cerrar la sesion se pierden los archivos
direccion = '/content'

In [ ]:
# ruta en local
direccion = 'local'

## 3. Carga de datos


In [ ]:
BATCH_SIZE = 64
train_loader, test_loader = load_mnist(BATCH_SIZE)

## 4. Definición y entrenamiento del autoencoder


In [ ]:
cuda = torch.cuda.is_available()
device = torch.device('cuda:0' if cuda else 'cpu')
print(device)
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 2),
            # ELEGIR FUNCIÓN DE ACTIVACIÓN
            nn.ReLU(inplace=True)
            #nn.Tanh()
        )

        self.decoder = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 784),
            nn.Tanh()

        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

def model_training(autoencoder, train_loader, epoch):
    loss_metric = nn.MSELoss()
    optimizer = torch.optim.Adam(autoencoder.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    epoch_t_loss = []

    autoencoder.train()
    for i, data in enumerate(train_loader):
        optimizer.zero_grad()
        images, _ = data
        images = images.view(images.size(0), -1)
        if cuda:
            images = images.to(device)
        outputs = autoencoder(images)
        loss = loss_metric(outputs, images)
        loss.backward()
        optimizer.step()
        if (i + 1) % LOG_INTERVAL == 0:
            print('Epoch [{}/{}] - Iter[{}/{}], MSE loss:{:.4f}'.format(
                epoch + 1, EPOCHS, i + 1, len(train_loader.dataset) // BATCH_SIZE, loss.item()
            ))
        epoch_t_loss.append(loss.item())
    return float(np.mean(epoch_t_loss))


def evaluation(autoencoder, test_loader, model_name, L1_loss = False):
    total_loss = 0
    loss_metric = nn.MSELoss()
    autoencoder.eval()
    for i, data in enumerate(test_loader):
        images, _ = data
        images = images.view(images.size(0), -1)
        if cuda:
            images = images.to(device)
        outputs = autoencoder(images)
        mse_loss = loss_metric(outputs, images)
        loss = mse_loss
        total_loss += loss.item() * len(images)
    avg_loss = total_loss / len(test_loader.dataset)

    print('\nAverage MSE Loss on Test set: {:.4f}'.format(avg_loss))

    global BEST_VAL
    if TRAIN_SCRATCH and avg_loss < BEST_VAL:
        BEST_VAL = avg_loss
        torch.save(autoencoder.state_dict(), os.path.join(direccion, model_name + ".pt"))
        print('Save Best Model in HISTORY\n')

    return float(avg_loss)


def plot_training_history(train_loss_hist, eval_loss_hist, title):
    x_seq = list(range(1, len(train_loss_hist) + 1))
    plt.figure(figsize=(10, 6))
    plt.plot(x_seq, train_loss_hist, label="Train reconstruction loss", linewidth=2)
    plt.plot(x_seq, eval_loss_hist, label="Validation reconstruction loss", linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


In [ ]:
autoencoder = Autoencoder()
autoencoder

In [ ]:
EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
SPARSE_REG = 1e-3
LOG_INTERVAL = 100
TRAIN_SCRATCH = True  # whether to train a model from scratch
BEST_VAL = float('inf')     # record the best val loss
MODEL_NAME = 'simple_autoencoder_2d_relu_40'

train_loss_hist = []
eval_loss_hist = []

if cuda:
    autoencoder.to(device)

if TRAIN_SCRATCH:
    for epoch in range(EPOCHS):
        starttime = datetime.datetime.now()
        train_loss = model_training(autoencoder, train_loader, epoch)
        endtime = datetime.datetime.now()
        print(f'Train a epoch in {(endtime - starttime).seconds} seconds')
        eval_loss = evaluation(autoencoder, test_loader, MODEL_NAME)
        train_loss_hist.append(train_loss)
        eval_loss_hist.append(eval_loss)
    print('Trainig Complete with best validation loss {:.4f}'.format(BEST_VAL))
    plot_training_history(train_loss_hist, eval_loss_hist, "Autoencoder 2D: reconstruction loss")

# una vez entrenado, que genere la imagen de ejemplo
autoencoder.load_state_dict(torch.load(os.path.join(direccion, MODEL_NAME + ".pt"), map_location = device))
autoencoder.cpu()
dataiter = iter(train_loader)
images, _ = next(dataiter)
images = images[:32]
outputs = autoencoder(images.view(images.size(0), -1))
plt.figure(figsize=(16,8))
plt.subplot(121)
plt.title('Original MNIST Images')
imshow(torchvision.utils.make_grid(images))
plt.subplot(122)
plt.title('Autoencoder Reconstruction')
imshow(torchvision.utils.make_grid(
        outputs.view(images.size(0), 1, 28, 28).data
))
plt.savefig(os.path.join(direccion, MODEL_NAME + ".png"))


## 5. Análisis del espacio latente

Comparar la separación entre clases y la compactación del espacio para ambas variantes entrenadas.


In [ ]:
# estamos generando un nuevo model, que copia el sequential del model original, la parte del encoder
pepe = autoencoder.encoder
pepe.load_state_dict(autoencoder.encoder.state_dict())

In [ ]:
pepe.to('cpu')

In [ ]:
# mapeo el laten space del test
# le pasamos un batch del TEST set y plotemaos la codificación del mismo
img, label = next(iter(test_loader))

# acá le pasamos TODO el batch del TEST al ENCODER, en las dimensiones adecuadas (las
# imagenes de 28x28 son vectorizadas).

latent = pepe(img.view(img.size(0), -1))
# vemos el tamaño de lo que obtuvimos
print('           batch x size')
latent.shape

In [ ]:
label.shape

### 5.1 Visualización 2D


In [ ]:
# ploteamos los resultados
fig, ax = plt.subplots(figsize=(6, 6))
col = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown',
    'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']

# obtenemos las coordenadas de cada cifra (codificación del encoder)
x_l, y_l= latent[:,0].detach().numpy(), latent[:,1].detach().numpy()

# obtenemos el label de cada cifra para colorear cada punto
color = label.detach().numpy()

# graficamos cada cifra en su espacio latente
for k in range(10):
  ax.scatter(x_l[color==k], y_l[color==k], c=col[k])

# Etiquetamos cada punto para mayor claridad
texts = [ax.text(x_l[i], y_l[i], str(txt.item())) for i, txt in enumerate(label)]

#ax.scatter(0, 0, c='black')
ax.legend(range(10))
plt.title('Latent space de 1 batch del test')

plt.grid()
plt.show()


In [ ]:
# para graficar una cifra específica

#############
k = 7     # cifra a graficar
############
# ploteamos los resultados
fig, ax = plt.subplots(figsize=(6, 6))
col = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown',
    'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']

# obtenemos las coordenadas de cada cifra (codificación del encoder)
x_l, y_l= latent[:,0].detach().numpy(), latent[:,1].detach().numpy()
lab = label.detach().numpy()

# graficamos cada cifra en su espacio latente
color = label.detach().numpy()


ax.scatter(x_l[color==k], y_l[color==k], c=col[k])

# Etiquetamos cada punto para mayor claridad
texts = [ax.text(x_l[lab==k][i], y_l[lab==k][i], txt) for i,
     txt in enumerate(lab[lab==k])]

ax.scatter(0, 0, c='black')
ax.legend(str(k))
plt.title('Latent space de 1 batch del test')

plt.grid()
plt.show()

## 6. Generación desde el espacio latente


In [ ]:
jose = autoencoder.decoder

In [ ]:
jose.load_state_dict(autoencoder.decoder.state_dict())

In [ ]:
# generamos el vector de entrada al DECODER
entrada = 0.1*torch.rand(2) + torch.tensor([2,1])
entrada

In [ ]:
# se lo pasamos al modelo DECODER
est = jose(entrada)

In [ ]:
# vemos le tamaño de la respuesta
est.shape

In [ ]:
# lo re-ordenamos y sacamos los valores del tensor (para deshacernos del gradiente)
imagen = est.view(1, 28, 28).data
imagen.shape

In [ ]:
plt.figure()
plt.subplot(111)
plt.title('Autoencoder Reconstruction')
plt.imshow(imagen.squeeze(), cmap="gray")
plt.show()

## 7. Mosaico de cifras generado desde el espacio latente

Consigna: ajustar el rango de `x` e `y` para que el mosaico recorra regiones representativas del espacio latente y produzca dígitos reconocibles.


In [ ]:
# HIPER-PARAMETROS A FIJAR
n = 20        # número de imágenes (digits del NMIST) a representar.
x_min = 0    # valor mínimo de la variable x del `latent space`.
x_max = 0     # valor mínimo de la variable x del `latent space`.
y_min = 0    # valor mínimo de la variable x del `latent space`.
y_max =  0    # valor mínimo de la variable x del `latent space`.


# Display a 2D manifold of the digits
digit_size = 28
figure = np.zeros((digit_size * n, digit_size * n))

# Rangos de evaluación del latent space
grid_x = np.linspace(x_min, x_max, n)
grid_y = np.linspace(y_min, y_max, n)

for i, yi in enumerate(grid_x):
    for j, xi in enumerate(grid_y):
        z_sample = torch.tensor([float(xi), float(yi)])
        # acá se llama al decoder y se le pasa las coordenadas
        x_decoded = jose(z_sample)
        digit = x_decoded.view(1, 28, 28).data
        figure[(n-1-i) * digit_size: (n-1-i+1) * digit_size, (j) * digit_size: (j+1) * digit_size] = digit


fig, ax = plt.subplots(figsize=(8, 8))
plt.imshow(figure, cmap="gray")
ax.set_xticks(np.linspace(0,figure.shape[0],n))
ax.set_yticks(np.linspace(0,figure.shape[1],n))
ax.set_xticklabels(np.round(grid_x, decimals=1))
ax.set_yticklabels(np.flip(np.round(grid_y, decimals=1)))
plt.show()
